
### Breakdown of a simple neural network

- x -- input
- wx -- weights
- bx -- bias
- A -- Activation function
- y - output

- z = w1.x + b1
- z' = A(z)
- y = W2.Z' + b2


- Loss function
- Backpropagation
- Optimizer


### Component of pytorch

- Base class of defining custom models: torch.nn.Module
- Fully connected (dense) layer: torch.nn.Linear
- Activation function: torch.nn.ReLU
- Optimiser: torch.optim
- Loss function: torch.nn.CrossEntropyLoss
- Loads data in batch: torch.utils.data.DataLoader

#### Different way to create neural network

1. Function: Flexible, harder
2. Sequential: nn.Sequential 

### Building a neural network

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

In [5]:
# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [11]:
input_dim = 4
hidden_layer = 64
num_classes = 2

In [20]:
# Make model with Functional API 
class SimpleNNFunction(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


In [21]:
model_func = SimpleNNFunction(input_dim, hidden_dim=hidden_layer, output_dim=num_classes)

In [22]:
model_func

SimpleNNFunction(
  (fc1): Linear(in_features=4, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=2, bias=True)
)

In [23]:
# Make model with Sequestion 
class SimpleNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), # input -> hidden layer
            nn.ReLU(),                        # non - linear activation function 
            nn.Linear(hidden_dim, output_dim) # hidden -> Output layer
        )

    def forward(self, x):
        return self.net(x)


In [24]:
model = SimpleNN(input_dim, hidden_dim=hidden_layer, output_dim=num_classes)

In [25]:
model

SimpleNN(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=2, bias=True)
  )
)

In [32]:
# sample data
X = torch.randn(100, 4)
y = torch.randint(0, 2, (100,))

# loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [31]:
y

tensor([1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1,
        0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0,
        1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1,
        1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0,
        1, 0, 1, 1])

In [34]:
X[:10]

tensor([[-0.6299, -0.3778, -0.6904,  1.2048],
        [ 1.6159,  0.6157, -0.0458,  0.8537],
        [-0.8335,  0.9013, -0.8245, -0.5511],
        [ 0.3393, -1.9987, -0.5908,  0.1451],
        [-1.1489, -1.8543, -0.0777, -0.6262],
        [-1.8483,  0.0584, -1.4110,  0.9279],
        [-0.7993, -1.4267,  0.8536, -0.8737],
        [-0.1819,  0.8702, -0.6357,  0.2849],
        [-0.0177, -1.1250,  0.6668,  0.1862],
        [ 0.2035,  1.7712, -1.0334,  0.6618]])

In [36]:
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)

In [43]:
# training
epochs = 100

for epoch in range(epochs):
    model.train() # training mode
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward() # backword propagation
    optimizer.step() # update all the weights

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1}] Loss: {loss.item() :.4f}")


Epoch [10] Loss: 0.4507
Epoch [20] Loss: 0.4463
Epoch [30] Loss: 0.4420
Epoch [40] Loss: 0.4378
Epoch [50] Loss: 0.4336
Epoch [60] Loss: 0.4294
Epoch [70] Loss: 0.4253
Epoch [80] Loss: 0.4209
Epoch [90] Loss: 0.4166
Epoch [100] Loss: 0.4124


In [58]:
# Evaluation
model.eval()

with torch.no_grad(): # stop gradient
    outputs = model(X)
    predicted = torch.argmax(outputs, dim=1)
    print(predicted)
    correct = (predicted == y).sum().item()
    print("Correct --> ", correct)
    total = len(y)
    print("Total --> ", total)
    accuracy = correct / total
    print(f"Evaluation accuracy: {accuracy*100:.2f}%")

tensor([1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0,
        0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0,
        0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0,
        1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1,
        1, 1, 0, 1])
Correct -->  83
Total -->  100
Evaluation accuracy: 83.00%
